# Parquet and ORC: Columnar Storage, Compression, and Speedups

**Module**: 08_data_sources_and_formats  
**Notebook**: 02_parquet_and_orc.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: Why Parquet and ORC are the gold standard for big data analytics: columnar storage, row-group statistics, dictionary encoding, and predicate pushdown.
- **Why does this topic exist?**: Switching from CSV to Parquet typically yields 5x-10x storage savings and 10x-100x query speedups.
- **What problem does it solve?**: Inefficient full-table scans on uncompressed text formats.
- **Where is it used in real-world Data Engineering?**: Enterprise silver/gold lakehouse tables.

---

## 2. Row vs Columnar Format
```
Row-Oriented (CSV):
[Row 0: 1001, 2026-01-03, North, Laptop] [Row 1: 1002, 2026-01-03, South, Phone]
(To read only'revenue', MUST scan all bytes across all columns!)

Columnar Format (Parquet):
[Column'order_id': 1001, 1002, ...]
[Column'region':   North, South, ...]  <-- Dictionary Encoded & Compressed!
(Reads ONLY the bytes of requested columns; skips unneeded row groups via min/max stats!)
```

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = (
    SparkSession.builder
    .appName("Parquet_ORC_Lab")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

In [ ]:
orders = spark.read.option("header", True).option("inferSchema", True).csv("/opt/spark-data/retail/orders.csv")
out_parquet = "/opt/spark-data/warehouse/orders_parquet"

# Write as Parquet
orders.write.mode("overwrite").parquet(out_parquet)
print("Written orders to Parquet successfully!")

# Read back and inspect
read_parquet = spark.read.parquet(out_parquet)
print(f"Parquet row count: {read_parquet.count()}")

---

## Experiment: Storage Size: CSV vs Parquet

- **1. What are we testing?**: Compare file size on disk between raw CSV and Snappy-compressed Parquet.
- **2. What do we expect?**: Parquet is 3x to 6x smaller than CSV.

In [ ]:
csv_size = os.path.getsize("/opt/spark-data/retail/orders.csv")
parquet_size = sum(os.path.getsize(os.path.join(out_parquet, f)) for f in os.listdir(out_parquet) if f.endswith(".parquet"))
print(f"Raw CSV Size:     {csv_size:,} bytes")
print(f"Parquet Size:     {parquet_size:,} bytes")
print(f"Compression Ratio: {csv_size / parquet_size:.2f}x smaller!")

**3. What happened?**: Parquet achieved massive compression savings.  
**4. Why did it happen?**: Columnar layout enables run-length encoding and dictionary compression.  
**5. Practical Engineering Lesson**: Never store historical data in CSV/JSON; always convert to Parquet.

---

## Practical Exercises

### Exercise 1 (Beginner): Read Parquet Subset
**Problem Statement**: Read only order_id and unit_price from Parquet.  
**Input / Context**: read_parquet

In [ ]:
# Write your solution for Exercise 1 here

### Exercise 2 (Intermediate): Verify Predicate Pushdown
**Problem Statement**: Filter Parquet on quantity > 10 and verify PushedFilters in plan.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 2 here

### Exercise 3 (Advanced): Snappy vs ZSTD Compression
**Problem Statement**: Write with compression codec'zstd'.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 3 here

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (Read Parquet Subset):
```python
spark.read.parquet(out_parquet).select('order_id','unit_price').show(3)
```

#### Solution 2 (Verify Predicate Pushdown):
```python
spark.read.parquet(out_parquet).filter('quantity > 10').explain()
```

#### Solution 3 (Snappy vs ZSTD Compression):
```python
orders.write.mode('overwrite').option('compression','zstd').parquet('/opt/spark-data/warehouse/orders_zstd')
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **Columnar Format**: Compresses identical types together; supports column pruning and predicate pushdown.
- **Standard**: Parquet is Spark's native format.

### Essential Technical Terms
- `Row Group`: Chunk of columnar data with min/max statistics.
- `Snappy`: Default fast compression codec.

### Common Engineering Mistakes
- **Using CSV in Production**: Wastes storage and kills query performance.

### Interview Questions on This Topic
- **Q: How does Parquet skip reading data?**
  - *A*: Through row group metadata storing min/max statistics for each column, allowing Spark to skip entire blocks if the filter range does not intersect.

### What to Study Next
Open 03_partitioned_data_lake.ipynb

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')